In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

In [ ]:
df = pd.read_csv('../data/df_n.csv')

df = df.drop(columns=['Unnamed: 0'])

df.columns

# 🚦 Paso 1: Elegir la variable objetivo

In [ ]:
target = 'churned'

# ✅ Paso 2: Seleccionar características (features)

Empieza con un conjunto de columnas que sean relevantes para predecir churn. Por ejemplo, podrías elegir:

Variables de actividad:

dias_inactivo, num_transactions, num_notifications, converted

Datos demográficos:

age, country, plan, channel, age_group

Otros indicadores útiles:

user_settings_crypto_unlocked, has_notification, has_device

Ejemplo inicial de features:

In [ ]:
features = [
    'birth_year', 'country', 'city', 'user_settings_crypto_unlocked', 'plan',
    'attributes_notifications_marketing_push', 'attributes_notifications_marketing_email',
    'num_contacts', 'num_referrals', 'num_successful_referrals',
    'brand_device', 'has_device', 'reason', 'channel', 'status', 'has_notification',
    'age', 'age_group'
]


# 🚨 Paso 3: Crear un dataset limpio por usuario único

In [ ]:
# Filtra solo usuarios con al menos una transacción
usuarios_activos = df[df['has_transaction'] == True]

# Quita duplicados para quedarte con un registro único por usuario
df_model = usuarios_activos.drop_duplicates(subset='user_id').copy()

df_model

# 🔎 Paso 4: Preparar variables categóricas y numéricas

In [ ]:
cat_features = ['country', 'plan', 'channel', 'age_group']
num_features = [f for f in features if f not in cat_features]

display(cat_features, num_features)

# 🧰 Paso 5: Construir un pipeline de preprocesamiento + modelo

In [ ]:

# 1) Identificar las columnas numéricas y categóricas
categorical_cols = df_model[features].select_dtypes(include='object').columns.tolist()
numerical_cols = df_model[features].select_dtypes(include='number').columns.tolist()

# 2) Construir preprocesador con ColumnTransformer
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numerical_cols)
])

# 3) Pipeline con preprocesamiento y regresión logística
model_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])


# 🧪 Paso 6: Separar train/test y entrenar

In [ ]:
X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model_pipeline.fit(X_train, y_train)

display(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# 📊 Paso 7: Evaluar desempeño

In [ ]:
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
